Correct warpcoefficient libraries such that templates can be generated with a color distribution matching the observed.

We here test whether the corrected templates calculated in warpcoeff_colorcorrect owrk

In [ ]:
import os, re, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#from scipy.optimize import curve_fit
#from scipy.stats import exponnorm
from pathlib import Path
from datetime import datetime
import sncosmo
import extinction
from scipy.optimize import brentq
import pickle

In [ ]:
sys.path.append('/Users/jnordin/github/ampelFeb25')

In [ ]:
from warpTemplate import WarpfitTemplateLoader

In [ ]:
template_class_id = 9

In [ ]:
# NOw you have to decide which warp class to fit against:
warpclasses = [
    'SN Ib', 'SN Ia-91bg', 'SN II', 'SN Ia-91T',
    'SN Ib/c', 'SN Ia-pec', 'SN IIP', 'SN Iax', 'SN Ic',
    'SLSN-II']

In [ ]:
print('Fitting to templates of class', warpclasses[template_class_id])

In [ ]:
# Help methods to get mean color for model
def get_latest_model_result(model_name, infile="warptemplate_color_fits.csv"):
    file = Path(infile)

    if not file.exists():
        raise FileNotFoundError(f"{infile} does not exist")

    # Load data
    df = pd.read_csv(file)

    if df.empty:
        return None

    # Ensure timestamp is datetime
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # Filter by model
    df_model = df[df["model"] == model_name]

    if df_model.empty:
        return None

    # Get latest entry
    latest_row = df_model.sort_values("timestamp", ascending=False).iloc[0]

    return latest_row.to_dict()

In [ ]:
model_colors = get_latest_model_result(warpclasses[template_class_id])

In [ ]:
model_colors

In [ ]:
target_color = model_colors['loc']

In [ ]:
# Now we know which peak color the warped templates should be shifted 

In [ ]:
# Parameters for fit template retrieval
# How to define templates?
# - How many templates per sn basis? 
#      * if 'all' it will return one copy of each template, 
#.     * if int it will return that many, drawn according to the template probability, 
#      * if -int it will return that many copies drawn from a uniform probabilitiy
#      Note: draws made with replacement, so multiple copies can be returned if int is larger than the available number of templates (often 3)
template_selection = 1    # Use the same number per SN to keep balance 
# - How many sn basis?
#.     * if 'all', take one of each
#.     * if an int, draw these randomly (with replacement)
#.     Note: how many templates are returned is decided by the above parameter.
snbasis_selection = 'all'

warpdir = '/Users/jnordin/data/models/sncosmo/warpmod'

In [ ]:
warploader = WarpfitTemplateLoader(warpdir)

In [ ]:
templates = warploader.get_templates(
    fitclass=warpclasses[template_class_id],
    exclude_input = [],    # Here we do not wish to exclude anything 
    template_selection=template_selection,
    snbasis_selection=snbasis_selection,
    random_seed=42,
    harmonize_colors=True
)

In [ ]:
# Now we wish to loop through all sncosmo templates and:
# - measure the template peak phase color
# - calculate which extinction would need to be applied to this template to achieve the class color 
# - create a modified version of the template warp coefficients that lead to the desired color
# - store the new version 

In [ ]:
for k, t in enumerate(templates):

    # This 
    mod = t['model']
    modid = mod.description

    peakcol = mod.bandmag(model_colors['color1'], 'ab', 0)-mod.bandmag(model_colors['color2'], 'ab', 0)
    mod.set(colcorrebv=0)
    oldcol = mod.bandmag(model_colors['color1'], 'ab', 0)-mod.bandmag(model_colors['color2'], 'ab', 0)

    if np.abs(target_color-peakcol)>0.01:
        print(k,modid, oldcol, peakcol, np.round(target_color-peakcol), target_color)
    if k%100==0:
        print('Check:', k,modid, oldcol, peakcol, np.round(target_color-peakcol), target_color)

